# 26b — part 2: the paired de-shortcut eval (GPU)

Three arms over the same 673 `fo_class` questions: same frame, same gold, different phrasing.
One checkpoint, no training. Pre-registration and the decision rule: `PLAN.md`.

🔴 **All three arms run in ONE process on ONE GPU**, including `original`. Reusing archived
answers would put the measured ~0.5% cross-GPU drift inside a paired delta read at ε = 0.05.

🔴 **Primary read is ID** (28 videos vs OOD's 10; effective n is videos, RULES §13), and it is
the half that populates the leaderboard's `pre_evaluation_score` today. `INCONCLUSIVE` on OOD is
expected and is not a finding.

In [ ]:
# ── config (inline; papermill-overridable) ───────────────────────────────────────────────────
SMOKE = True                 # True → N_SMOKE questions per arm, to prove the wiring
N_SMOKE = 12

# The arm that owns object_recognition_ID at epoch 3, per the selection rule declared
# 2026-07-30. A3_vitlr did NOT displace it: its paired delta vs A2 excludes zero on the
# WRONG side (ALL -0.0293), and the rule's intent is "significantly better", not "significant".
MERGED = "/workspace/repo/experiments/21-recipe-sweep/runs/21_lr_2e4_v1/merged/checkpoint-2703"
CKPT_LABEL = "21_lr_2e4_v1/ep3"

REPO = "/workspace/repo"
DATA_ROOT = "/workspace/orena-data"
OUT_DIR = "/workspace/repo/experiments/26-deshortcut-eval/runs"
ARMS_CSV = "/workspace/repo/experiments/26-deshortcut-eval/tables/part2_arms.csv"
EPSILON = 0.05               # pre-declared in PLAN.md, looser than training A/Bs on purpose

In [ ]:
# 🔴 HF_HOME before the offline flags mean anything: the SDK judge is cached at
# /workspace/hf_cache, NOT at the default ~/.cache/huggingface.
import os

os.environ.setdefault("HF_HOME", "/workspace/hf_cache")

import sys
from pathlib import Path

sys.path.insert(0, f"{REPO}/src")
sys.path.insert(0, f"{REPO}/experiments/26-deshortcut-eval/_models")

import pandas as pd

from deshortcut import ARMS

arms_df = pd.read_csv(ARMS_CSV)
assert set(arms_df["arm"]) == set(ARMS), f"arms.csv carries {set(arms_df['arm'])}"

# qID → question, per arm. The rewriter is a pure lookup: the transformation itself already
# ran (and was gated) in 26_deshortcut_eval.ipynb, so nothing is re-derived on the GPU box.
QMAP = {a: dict(g[["qID", "question"]].values) for a, g in arms_df.groupby("arm")}
QIDS = set(arms_df.loc[arms_df["arm"] == "original", "qID"])

print(f"checkpoint  {CKPT_LABEL}")
print(f"stratum     {len(QIDS)} questions x {len(ARMS)} arms")
print(f"mode        {'SMOKE N=' + str(N_SMOKE) if SMOKE else 'FULL RUN'}")

In [ ]:
# ── G-ARM: the rewriter must actually rewrite ───────────────────────────────────────────────
#
# The failure this catches is silent and fatal: a lookup that misses every qID leaves the
# question untouched, all three arms answer the SAME text, and the perfect null that follows
# reads like the cleanest possible "the margin is vision". RULES §7 — gates raise.
for arm in ARMS:
    missing = QIDS - set(QMAP[arm])
    assert not missing, f"{arm}: {len(missing)} qIDs absent from the map"

n_diff = {a: sum(QMAP[a][q] != QMAP["original"][q] for q in QIDS) for a in ARMS}
assert n_diff["original"] == 0, "the control arm is not the corpus question"
for arm in ("premise_dropped", "set_framed"):
    assert n_diff[arm] == len(QIDS), f"{arm}: only {n_diff[arm]}/{len(QIDS)} differ from control"

print("G-ARM ok —", {a: f"{n}/{len(QIDS)} rewritten" for a, n in n_diff.items()})

In [ ]:
# ── the three runs ──────────────────────────────────────────────────────────────────────────
import time

from frame.config import BaselineConfig
from frame.run import run_baseline

reports = {}
for arm in ARMS:
    qmap = QMAP[arm]
    cfg = BaselineConfig(
        data_root=Path(DATA_ROOT),
        model_path=Path(MERGED),
        out_dir=Path(OUT_DIR),
        run_name=f"26_part2_{arm}" + ("_smoke" if SMOKE else ""),
        n_eval=N_SMOKE if SMOKE else None,
        question_rewriter=lambda qid, q, _m=qmap: _m.get(qid, q),
    )
    t0 = time.time()
    reports[arm] = run_baseline(cfg, qid_filter=QIDS)
    print(f"\n=== {arm}: done in {(time.time() - t0) / 60:.1f} min\n")

In [ ]:
# ── the read: paired delta + TOST, primary on ID ────────────────────────────────────────────
#
# paired_delta_ci takes ONE frame carrying qID, video and both correctness columns, and
# returns delta = b - a. So b = the de-named arm and a = the control, which makes the sign
# read directly as "de-named minus original" — the quantity PLAN.md's rule is written about.
from frame.metrics import equivalence_verdict, paired_delta_ci

SUFFIX = "_smoke" if SMOKE else ""


def _scored(arm: str) -> pd.DataFrame:
    df = pd.read_csv(Path(OUT_DIR) / f"26_part2_{arm}{SUFFIX}" / "results.csv")
    df["distribution"] = df["qID"].map(lambda q: "OOD" if q.split("__", 1)[0] == "heico" else "ID")
    return df[["qID", "video", "correctness", "distribution"]]


control = _scored("original")
rows = []
for arm in ("premise_dropped", "set_framed"):
    paired = control.merge(_scored(arm)[["qID", "correctness"]], on="qID",
                           suffixes=("_a", "_b"))
    paired = paired.rename(columns={"correctness_a": "correct_a", "correctness_b": "correct_b"})
    assert len(paired) == len(control), "arms are not scored on the same questions"
    for cell in ("ID", "OOD", "ALL"):
        sl = paired if cell == "ALL" else paired[paired["distribution"] == cell]
        if sl.empty:
            continue
        ci = paired_delta_ci(sl)
        eq = equivalence_verdict(ci, epsilon=EPSILON)
        rows.append({"arm": arm, "cell": cell, "n": len(sl), "videos": sl["video"].nunique(),
                     **ci, "verdict": eq["verdict"], "epsilon_min": eq.get("epsilon_min")})

read = pd.DataFrame(rows)
display(read)
if not SMOKE:
    read.to_csv(f"{REPO}/experiments/26-deshortcut-eval/tables/part2_read.csv", index=False)
    print("wrote tables/part2_read.csv")

## Reading it (fixed before any number, `PLAN.md`)

Primary arm `premise_dropped`, primary cell **ID**:

| paired delta | reading |
|---|---|
| CI excludes 0 **and** delta ≤ −0.10 | 🔴 the margin is substantially phrasing — `object_recognition` is overstated and every ladder comparison leaning on it needs re-reading |
| `EQUIVALENT` at ε = 0.05 | 🟢 the margin survives de-naming: it is vision, and part 1's gap was difficulty, not exploitation |
| `INCONCLUSIVE` | underpowered — report as such, do **not** read as a pass. `epsilon_min` is the honest power statement: no ε below it could ever have concluded |

`set_framed` is secondary: it removes the premise *and* changes to set framing, so it cannot
attribute an effect to one variable. Its own yield is different — over-listing on frames whose
gold is a single class would be the individuation deficit showing up inside `fo_class`, not
only in `number`.